# In google colab

In [1]:
# !git clone https://github.com/Thanh-Thanh-Thanh/thanh.git
# !pip install -r /content/thanh/requirements.txt

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# Import lib

In [3]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
import sys
sys.path.append('../') # comment if workspace
# sys.path.append('/content/thanh') # if colab
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq, select_mcs, calculate_tb_size
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import *
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, \
    BinarySource, config_parser, fft_size_return
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper
from sionna.fec.ldpc import LDPC5GEncoder

from sionna.nr.my_abc import *

In [4]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm import tqdm
import itertools
import io
import h5py
import random

## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [5]:
# from dataclasses import dataclass, field
# from typing import List

# @dataclass
# class SystemConfig:
#     NCellId: int = 246
#     FrequencyRange: int = 1
#     BandWidth: int = 100
#     Numerology: int = 1
#     CpType: int = 0
#     NTxAnt: int = 1
#     NRxAnt: int = 8
#     BwpNRb: int = 273
#     BwpRbOffset: int = 0
#     harqProcFlag: int = 0
#     nHarqProc: int = 1
#     rvSeq: int = 0


# @dataclass
# class UeConfig:
#     TransformPrecoding: int = 0
#     Rnti: int = 20002
#     nId: int = 246
#     CodeBookBased: int = 0
#     DmrsPortSetIdx: List[int] = field(default_factory=lambda: [0])  # FIXED
#     NLayers: int = 1
#     NumDmrsCdmGroupsWithoutData: int = 2
#     Tpmi: int = 0
#     FirstSymb: int = 0
#     NPuschSymbAll: int = 14
#     RaType: int = 1
#     FirstPrb: int = 31
#     NPrb: int = 4
#     FrequencyHoppingMode: int = 0
#     McsTable: int = 0
#     Mcs: int = 3
#     ILbrm: int = 0
#     nScId: int = 0
#     NnScIdId: int = 246
#     DmrsConfigurationType: int = 0
#     DmrsDuration: int = 1
#     DmrsAdditionalPosition: int = 1
#     PuschMappingType: int = 0
#     DmrsTypeAPosition: int = 3
#     HoppingMode: int = 0
#     NRsId: int = 0
#     Ptrs: int = 0
#     ScalingFactor: int = 0
#     OAck: int = 0
#     IHarqAckOffset: int = 11
#     OCsi1: int = 0
#     ICsi1Offset: int = 7
#     OCsi2: int = 0
#     ICsi2Offset: int = 0
#     NPrbOh: int = 0
#     nCw: int = 1
#     TpPi2Bpsk: int = 0

# @dataclass
# class MyConfig:
#     Sys: SystemConfig
#     Ue: List[UeConfig]
#     Num_tx: int = 1
#     Num_rx: int = 1
#     Carrier_frequency: float = 2.55e9  # Carrier frequency in Hz

# # Example usage
# My_Config = MyConfig(SystemConfig(), [UeConfig()])


In [6]:
# class MyPUSCHConfig(PUSCHConfig):
#     def __init__(self, My_Config: MyConfig):
#         self.My_Config = My_Config
#         super().__init__(
#             carrier_config=CarrierConfig(
#                 n_cell_id=My_Config.Sys.NCellId,
#                 cyclic_prefix="normal" if ~My_Config.Sys.CpType else "extended",
#                 subcarrier_spacing=15*(2**My_Config.Sys.Numerology),
#                 n_size_grid=My_Config.Sys.BwpNRb,
#                 n_start_grid=My_Config.Sys.BwpRbOffset,
#                 slot_number=4,
#                 frame_number=0
#             ),
#             pusch_dmrs_config=PUSCHDMRSConfig(
#                 config_type=My_Config.Ue[0].DmrsConfigurationType + 1,
#                 length=My_Config.Ue[0].DmrsDuration,
#                 additional_position=My_Config.Ue[0].DmrsAdditionalPosition,
#                 dmrs_port_set=My_Config.Ue[0].DmrsPortSetIdx,
#                 n_id=My_Config.Ue[0].NnScIdId,
#                 n_scid=My_Config.Ue[0].nScId,
#                 num_cdm_groups_without_data=My_Config.Ue[0].NumDmrsCdmGroupsWithoutData,
#                 type_a_position=My_Config.Ue[0].DmrsTypeAPosition
#             ),
#             tb_config=TBConfig(
#                 channel_type='PUSCH',
#                 n_id=My_Config.Ue[0].nId,
#                 mcs_table=My_Config.Ue[0].McsTable + 1,
#                 mcs_index=My_Config.Ue[0].Mcs
#             ),
#             mapping_type='A' if ~My_Config.Ue[0].PuschMappingType else 'B',
#             n_size_bwp=My_Config.Sys.BwpNRb,
#             n_start_bwp=My_Config.Sys.BwpRbOffset,
#             num_layers=My_Config.Ue[0].NLayers,
#             num_antenna_ports=len(My_Config.Ue[0].DmrsPortSetIdx),
#             precoding='non-codebook' if ~My_Config.Ue[0].CodeBookBased else 'codebook',
#             tpmi=My_Config.Ue[0].Tpmi,
#             transform_precoding=False if ~My_Config.Ue[0].TransformPrecoding else True,
#             n_rnti=My_Config.Ue[0].Rnti,
#             symbol_allocation=[My_Config.Ue[0].FirstSymb,My_Config.Ue[0].NPuschSymbAll]
#         )
#     @property
#     def first_resource_block(self):
#         """
#         :class:`~sionna.nr.CarrierConfig` : Carrier configuration
#         """
#         return self.My_Config.Ue[0].FirstPrb

#     @property
#     def first_subcarrier(self):
#         """
#         :class:`~sionna.nr.CarrierConfig` : Carrier configuration
#         """
#         return 12*self.first_resource_block

#     @property
#     def num_resource_blocks(self):
#         """
#         int, read-only : Number of allocated resource blocks for the
#             PUSCH transmissions.
#         """
#         return self.My_Config.Ue[0].NPrb

#     @property
#     def dmrs_grid(self):
#         # pylint: disable=line-too-long
#         """
#         complex, [num_dmrs_ports, num_subcarriers, num_symbols_per_slot], read-only : Empty
#             resource grid for each DMRS port, filled with DMRS signals

#             This property returns for each configured DMRS port an empty
#             resource grid filled with DMRS signals as defined in
#             Section 6.4.1.1 [3GPP38211]. Not all possible options are implemented,
#             e.g., frequency hopping and transform precoding are not available.

#             This property provides the *unprecoded* DMRS for each configured DMRS port.
#             Precoding might be applied to map the DMRS to the antenna ports. However,
#             in this case, the number of DMRS ports cannot be larger than the number of
#             layers.
#         """
#         # Check configuration
#         self.check_config()

#         # Configure DMRS ports set if it has not been set
#         reset_dmrs_port_set = False
#         if len(self.dmrs.dmrs_port_set)==0:
#             self.dmrs.dmrs_port_set = list(range(self.num_layers))
#             reset_dmrs_port_set = True

#         # Generate empty resource grid for each port
#         a_tilde = np.zeros([len(self.dmrs.dmrs_port_set),
#                             self.num_subcarriers,
#                             self.carrier.num_symbols_per_slot],
#                             dtype=complex)
#         first_subcarrier = self.first_subcarrier
#         num_subcarriers = self.num_subcarriers

#         # For every l_bar
#         for l_bar in self.l_bar:

#             # For every l_prime
#             for l_prime in self.l_prime:

#                 # Compute c_init
#                 l = l_bar + l_prime
#                 c_init = self.c_init(l)
#                 # Generate RNG
#                 c = generate_prng_seq(first_subcarrier + num_subcarriers, c_init=c_init)
#                 c = c[first_subcarrier:]

#                 # Map to QAM
#                 r = 1/np.sqrt(2)*((1-2*c[::2]) + 1j*(1-2*c[1::2]))

#                 # For every port in the dmrs port set
#                 for j_ind, _ in enumerate(self.dmrs.dmrs_port_set):

#                     # For every n
#                     for n in self.n:

#                         # For every k_prime
#                         for k_prime in [0, 1]:

#                             if self.dmrs.config_type==1:
#                                 k = 4*n + 2*k_prime + \
#                                     self.dmrs.deltas[j_ind]
#                             else: # config_type == 2
#                                 k = 6*n + k_prime + \
#                                     self.dmrs.deltas[j_ind]

#                             a_tilde[j_ind, k, self.l_ref+l] = \
#                                 r[2*n + k_prime] * \
#                                 self.dmrs.w_f[k_prime][j_ind] * \
#                                 self.dmrs.w_t[l_prime][j_ind]

#         # Amplitude scaling
#         a = self.dmrs.beta*a_tilde

#         # Reset DMRS port set if it was not set
#         if reset_dmrs_port_set:
#             self.dmrs.dmrs_port_set = []

#         return a
# My_Config = MyConfig(SystemConfig(), [UeConfig()])
# Pusch_Config = MyPUSCHConfig(My_Config)
# Pusch_Config.show()

In [7]:
"""
frame_number, n_cell_id: No change
n_rnti, tb.n_id: Change in tb_encoder, but not in dmrs (same c -> same x)
slot_number, dmrs.n_id, dmrs.n_scid, mcs: No change in tb_encoder (same b -> same c), but in dmrs
"""

'\nframe_number, n_cell_id: No change\nn_rnti, tb.n_id: Change in tb_encoder, but not in dmrs (same c -> same x)\nslot_number, dmrs.n_id, dmrs.n_scid, mcs: No change in tb_encoder (same b -> same c), but in dmrs\n'

In [8]:
# PuschRecord = namedtuple("PuschRecord", [
#     "nSFN", "nSlot", "nPDU", "nGroup", "nUlsch", "nUlcch", "nRachPresent",
#     "nRNTI", "nUEId", "nBWPSize", "nBWPStart", "nSubcSpacing", "nCpType", "nULType",
#     "nMcsTable", "nMCS", "nTransPrecode", "nTransmissionScheme", "nNrOfLayers",
#     "nPortIndex", "nNid", "nSCID", "nNIDnSCID", "nNrOfAntennaPorts",
#     "nVRBtoPRB", "nPMI", "nStartSymbolIndex", "nNrOfSymbols", "nResourceAllocType",
#     "nRBStart", "nRBSize", "nTBSize", "nRV", "nHARQID", "nNDI", "nMappingType",
#     "nDMRSConfigType", "nNrOfCDMs", "nNrOfDMRSSymbols", "nDMRSAddPos",
#     "nPTRSPresent", "nAck", "nAlphaScaling", "nBetaOffsetACKIndex", "nCsiPart1",
#     "nBetaOffsetCsiPart1Index", "nCsiPart2", "nBetaOffsetCsiPart2Index",
#     "nTpPi2BPSK", "nTPPuschID", "nRxRUIdx", "nUE", "nPduIdx",

#     # New fields for channel and filenames
#     "Dmrs_mask",
#     "Channel_model", "Speed", "Delay_spread", "Esno_db",
#     "Data_filename","Data_dirname"
#     ]
# )

# def save_pickle(data, parent_name, group_name):
#     """Saves data to a pickle file."""
#     def save_to_pickle(data, filename):
#         with open(filename, "wb") as f:
#             pickle.dump(data, f)
#     b, c, y = data
#     save_to_pickle(b.numpy(), f'{parent_name}/{group_name}.b.pkl')
#     save_to_pickle(c.numpy(), f'{parent_name}/{group_name}.c.pkl')
#     save_to_pickle(y.numpy(), f'{parent_name}/{group_name}.y.pkl')

# def save_hdf5(data, parent_name, group_name):
#     b, c, y = data
#     with h5py.File(f"{parent_name}.hdf5", "a") as hf:
#         hf.create_dataset(f"{group_name}_b", data=b.numpy())
#         hf.create_dataset(f"{group_name}_c", data=c.numpy())
#         hf.create_dataset(f"{group_name}_y", data=y.numpy())


In [9]:
# # class MySimulator():
# #     def __init__(self, pusch_config: MyPUSCHConfig):

# #         self.Num_rx = pusch_config.My_Config.Num_rx
# #         self.Num_tx = pusch_config.My_Config.Num_tx
    
# #         tb_size = pusch_config.tb_size
# #         num_coded_bits = pusch_config.num_coded_bits
# #         target_coderate = pusch_config.tb.target_coderate
# #         num_bits_per_symbol = pusch_config.tb.num_bits_per_symbol

# #         num_layers = pusch_config.num_layers
# #         n_rnti = pusch_config.n_rnti
# #         n_id = pusch_config.tb.n_id

# #         self.Binary_Source = BinarySource(dtype=tf.float32)
# #         self.TB_Encoder = TBEncoder(target_tb_size=tb_size,
# #                             num_coded_bits=num_coded_bits,
# #                             target_coderate=target_coderate,
# #                             num_bits_per_symbol=num_bits_per_symbol,
# #                             num_layers=num_layers,
# #                             n_rnti=n_rnti,
# #                             n_id=n_id,
# #                             channel_type="PUSCH",
# #                             codeword_index=0,
# #                             use_scrambler=True,
# #                             verbose=False,
# #                             output_dtype=tf.float32)
        
# #         self.Constellation_Mapper = Mapper("qam", num_bits_per_symbol, dtype=tf.complex64)

# #         self.Layer_Mapper = LayerMapper(num_layers=num_layers, dtype=tf.complex64)
    
# #         self.Pilot_Pattern = PUSCHPilotPattern([pusch_config], dtype=tf.complex64)

# #         num_subcarriers = pusch_config.num_subcarriers
# #         subcarrier_spacing = pusch_config.carrier.subcarrier_spacing*1e3
# #         fft_size = num_subcarriers
# #         cp_length = min(num_subcarriers, 288)
# #         guard_subcarriers = (0,0)
# #         # Define the resource grid.
# #         resource_grid = ResourceGrid(
# #             num_ofdm_symbols=14,
# #             fft_size=fft_size,
# #             subcarrier_spacing=subcarrier_spacing,
# #             num_tx=self.Num_tx,
# #             num_streams_per_tx=1,
# #             cyclic_prefix_length=cp_length,
# #             num_guard_carriers=guard_subcarriers,
# #             dc_null=False,
# #             pilot_pattern=self.Pilot_Pattern,
# #             dtype=tf.complex64
# #         )

# #         self.Resource_Grid_Mapper = ResourceGridMapper(resource_grid, dtype=tf.complex64)        
        
# #         self.AWGN = AWGN()

 
# #         self.Channel_Estimator = PUSCHLSChannelEstimator(
# #                         resource_grid,
# #                         pusch_config.dmrs.length,
# #                         pusch_config.dmrs.additional_position,
# #                         pusch_config.dmrs.num_cdm_groups_without_data,
# #                         interpolation_type='nn',
# #                         dtype=tf.complex64)

# #         rxtx_association = np.ones([self.Num_rx, self.Num_tx], bool)
# #         stream_management = StreamManagement(rxtx_association, pusch_config.num_layers)
# #         self.Mimo_Detector = LinearDetector("lmmse", "bit", "maxlog", resource_grid, stream_management,
# #                                     "qam", pusch_config.tb.num_bits_per_symbol, dtype=tf.complex64)
        

# #         self.Layer_Demapper = LayerDemapper(self.Layer_Mapper, num_bits_per_symbol=num_bits_per_symbol)
# #         self.TB_Decode = TBDecoder(self.TB_Encoder, output_dtype=tf.float32)

# #         self.tb_size = tb_size
# #         self.resource_grid = resource_grid
# #         self.pusch_config = pusch_config
        
# #     def update_pilots(self, pilots):
# #         self.Resource_Grid_Mapper._resource_grid.pilot_pattern.pilots = pilots

# #         self.Channel_Estimator = PUSCHLSChannelEstimator(
# #                         self.Resource_Grid_Mapper._resource_grid,
# #                         self.pusch_config.dmrs.length,
# #                         self.pusch_config.dmrs.additional_position,
# #                         self.pusch_config.dmrs.num_cdm_groups_without_data,
# #                         interpolation_type='nn',
# #                         dtype=tf.complex64)

# #         rxtx_association = np.ones([self.Num_rx, self.Num_tx], bool)
# #         stream_management = StreamManagement(rxtx_association, self.pusch_config.num_layers)
# #         self.Mimo_Detector = LinearDetector("lmmse", "bit", "maxlog", self.Resource_Grid_Mapper._resource_grid, stream_management,
# #                                     "qam", self.pusch_config.tb.num_bits_per_symbol, dtype=tf.complex64)

# #     def sim(self, batch_size, channel_model, no_scaling, from_binary_source=True, gen_seed=2004*10+4):
# #         if from_binary_source:
# #             b = self.Binary_Source([batch_size, self.Num_tx, self.tb_size])
# #         else:
# #             b = tf.reshape(tf.constant(generate_prng_seq(batch_size * self.Num_tx * self.tb_size, gen_seed), dtype=tf.float32), [batch_size, self.Num_tx, self.tb_size])
# #         c = self.TB_Encoder(b)
# #         x_map = self.Constellation_Mapper(c)
# #         x_layer = self.Layer_Mapper(x_map)
# #         x = self.Resource_Grid_Mapper(x_layer)

# #         y, h = channel_model(x)
# #         no = no_scaling * tf.math.reduce_variance(y)

# #         y = self.AWGN([y, no])
        
# #         return b, c, y
    
# #     def rec(self, y):
# #         no_ = 0.001
# #         h_hat, err_var = self.Channel_Estimator([y, no_])
# #         llr_det = self.Mimo_Detector([y, h_hat, err_var, no_])
# #         llr_layer = self.Layer_Demapper(llr_det)
# #         b_hat, tb_crc_status = self.TB_Decode(llr_layer)

# #         return b_hat, llr_det, tb_crc_status

# ue_antenna = Antenna(polarization="single",
#                 polarization_type="V",
#                 antenna_pattern="38.901",
#                 carrier_frequency=2.55e9)

# gnb_array = AntennaArray(num_rows=1,
#                         num_cols=8//2,
#                         polarization="dual",
#                         polarization_type="cross",
#                         antenna_pattern="38.901",
#                         carrier_frequency=2.55e9)

# channel_model = CDL(model = 'C',
#                             delay_spread = 150*1e-9,
#                             carrier_frequency = 2.55e9,
#                             ut_array = ue_antenna,
#                             bs_array = gnb_array,
#                             direction = 'uplink',
#                             min_speed = 1,
#                             max_speed = 1)
# simulator = MySimulator(Pusch_Config)

# channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
#                                     add_awgn=False, normalize_channel=True, return_channel=True)



In [10]:
# %timeit b, c, y, h = simulator.sim(1, channel, 0.5, True); _,_,crc = simulator.rec(y); print(crc)

In [ ]:
def generate_data(name: str,
        data_dir: str,
        pusch_configs: List[MyPUSCHConfig],
        channel_scenarios: List[str],
        esno_dbs: List[float],
        slots: List[int],
        save_dataset: str = None):
    assert save_dataset in [None, 'hdf5', 'pickle']
    """set up save directory"""
    pusch_records=[]
    parquet_dir = f'{data_dir}/parquet'
    if save_dataset: os.makedirs(parquet_dir, exist_ok=True)
    hdf5_dir = f'{data_dir}/hdf5'
    if save_dataset == 'hdf5': os.makedirs(hdf5_dir, exist_ok=True)
    pickle_dir = f'{data_dir}/pickle'
    if save_dataset == 'pickle': os.makedirs(pickle_dir, exist_ok=True)

    """set up for per slot"""
    len_per_case = len(slots)

    total_iterations = len(pusch_configs) * len(esno_dbs) * len_per_case


    """generate ..."""
    with tqdm(total=total_iterations, desc="Generating Data") as pbar:
        for config_idx, pusch_config in enumerate(pusch_configs):
            # num_rx = pusch_config.My_Config.Num_rx # 1
            # num_tx = pusch_config.My_Config.Num_tx # 1
            # num_streams_per_tx = pusch_config.num_antenna_ports # 1
           
            Pusch_Pilots = {}
            Pusch_Slots = [4,5,14,15]
            for n, slot in enumerate(Pusch_Slots):
                pusch_config_i = pusch_config.clone()
                pusch_config_i.carrier.slot_number = slot
                pilot_pattern_i = PUSCHPilotPattern([pusch_config_i], dtype=tf.complex64)
                Pusch_Pilots[slot] = pilot_pattern_i.pilots
            len_pusch = len(Pusch_Slots)

            """set up for all case"""
            carrier_frequency = pusch_config.My_Config.Carrier_frequency
            num_rx_ant = pusch_config.My_Config.Sys.NRxAnt

            ue_antenna = Antenna(polarization="single",
                            polarization_type="V",
                            antenna_pattern="38.901",
                            carrier_frequency=carrier_frequency)

            gnb_array = AntennaArray(num_rows=1,
                                    num_cols=num_rx_ant//2,
                                    polarization="dual",
                                    polarization_type="cross",
                                    antenna_pattern="38.901",
                                    carrier_frequency=carrier_frequency)


            simulator = MySimulator(pusch_config)

            channel_scenario = random.choice(channel_scenarios)
            """channel_scenario form: {CDL}-{A|B|C|D|E}-{delay_spread (ns)}-{speed (m/s)}
                                or {Umi|Uma}-{low|high}-[OnPL]-[OnSF]-{delay_spread (ns)}-{speed (m/s)}

                Ex: CDL-A-150-10"""

            chn_scn = channel_scenario.split('-')
            
            model = chn_scn[1]
            channel = chn_scn[0]
            enable_pl = True if 'OnPL' in chn_scn else False # Umi/Uma enable pathloss
            enable_sf = True if 'OnSF' in chn_scn else False # Umi/Uma enable shadow fading

            speed = float(chn_scn[-1])
            delay_spread = float(chn_scn[-2])
            

            if 'CDL' == channel:
                channel_model = CDL(model = model,
                                        delay_spread = delay_spread*1e-9,
                                        carrier_frequency = carrier_frequency,
                                        ut_array = ue_antenna,
                                        bs_array = gnb_array,
                                        direction = "uplink",
                                        min_speed = speed,
                                        max_speed = speed)

            else:
                if 'Umi' == channel:
                    channel_model = UMi(carrier_frequency = carrier_frequency,
                                        o2i_model = model,
                                        ut_array = ue_antenna,
                                        bs_array = gnb_array,
                                        direction = "uplink",
                                        enable_pathloss = enable_pl,
                                        enable_shadow_fading = enable_sf)
                elif 'Uma' == channel:
                    channel_model = UMa(carrier_frequency = carrier_frequency,
                                        o2i_model = model,
                                        ut_array = ue_antenna,
                                        bs_array = gnb_array,
                                        direction = "uplink",
                                        enable_pathloss = enable_pl,
                                        enable_shadow_fading = enable_sf)

            simulator = MySimulator(pusch_config)

            channel_i = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)
            
            for esno_db in esno_dbs:
                no_scaling = pow(10., -esno_db / 10.)
                if channel in ['Umi', 'Uma']:
                    channel_i._cir_sampler.set_topology(*gen_topology(1,1,channel.lower(),min_ut_velocity=speed, max_ut_velocity=speed))

                for n,slot in enumerate(slots):
                    status_str = f"(config {config_idx} | channel {channel_scenario} | {esno_db} dB | slot {slot} | Sample: {n+1}/{len_per_case}"
                    pbar.set_description(status_str)
                    simulator.pusch_config.carrier.slot_number = slot
                    simulator.update_pilots(Pusch_Pilots[slot])
                    # print(tf.math.reduce_sum(simulator.pusch_config.dmrs_grid, [0,1]))
                    
                    b, c, y= simulator.sim(1, channel_i, no_scaling, return_channel=False)
                    # print(tf.math.reduce_sum(x, [0,1,2,4]))
                    
                    # b_hat, c_soft, crc = simulator.rec(y)
                    # print(crc)

                    assert b.shape[0] == b.shape[1] == c.shape[0] == c.shape[1] == y.shape[0] == y.shape[1] == 1
                    b = tf.cast(b, dtype=tf.uint8)[0][0]
                    c = tf.cast(c, dtype=tf.uint8)[0][0]
                    r = tf.transpose(tf.constant(simulator.pusch_config.dmrs_grid, dtype=y.dtype), [0, 2, 1])
                    y =  y[0][0]

                    timestamp = datetime.now().strftime("%Y%m%d%H%M%S%f")


                    # print(y, r)
                    if save_dataset == 'hdf5':
                        save_hdf5([b,c,y,r], f'{hdf5_dir}/{name}', timestamp)
                    if save_dataset == 'pickle':
                        save_pickle([b,c,y,r], f'{pickle_dir}/{name}', timestamp)

                    
                    
                    pusch_records.append(PuschRecord(
                                        nPhyCellId=pusch_config.carrier.n_cell_id,
                                        nSFN=(n // len_pusch) % 1023,
                                        nSlot=slot,
                                        nPDU=1,
                                        nGroup=1,
                                        nUlsch=1,
                                        nUlcch=0,
                                        nRachPresent=0,
                                        nRNTI=pusch_config.n_rnti,
                                        nUEId=0,
                                        nBWPSize=pusch_config.n_size_bwp,
                                        nBWPStart=pusch_config.n_start_bwp,
                                        nSubcSpacing=pusch_config.carrier.mu,
                                        nCpType=pusch_config.My_Config.Sys.CpType,
                                        nULType=0,
                                        nMcsTable=pusch_config.tb.mcs_table - 1,
                                        nMCS=pusch_config.tb.mcs_index,
                                        nTransPrecode=pusch_config.My_Config.Ue[0].TransformPrecoding,
                                        nTransmissionScheme=pusch_config.My_Config.Ue[0].CodeBookBased,
                                        nNrOfLayers=pusch_config.num_layers,
                                        nPortIndex=pusch_config.dmrs.dmrs_port_set,
                                        nNid=pusch_config.tb.n_id,
                                        nSCID=pusch_config.dmrs.n_scid,
                                        nNIDnSCID=pusch_config.dmrs.n_id[0],
                                        nNrOfAntennaPorts=pusch_config.My_Config.Sys.NRxAnt,
                                        nVRBtoPRB=0,
                                        nPMI=pusch_config.My_Config.Ue[0].Tpmi,
                                        nStartSymbolIndex=pusch_config.symbol_allocation[0],
                                        nNrOfSymbols=pusch_config.symbol_allocation[1],
                                        nResourceAllocType=1,
                                        nDMRSTypeAPos=pusch_config.dmrs.type_a_position,
                                        nRBStart=pusch_config.first_resource_block,
                                        nRBSize=pusch_config.num_resource_blocks,
                                        nTBSize=(pusch_config.tb_size//8),
                                        nRV=pusch_config.My_Config.Sys.rvSeq,
                                        nHARQID=n % 16,
                                        nNDI=1,
                                        nMappingType=pusch_config.My_Config.Ue[0].PuschMappingType,
                                        nDMRSConfigType=pusch_config.My_Config.Ue[0].DmrsConfigurationType,
                                        nNrOfCDMs=pusch_config.dmrs.num_cdm_groups_without_data,
                                        nNrOfDMRSSymbols=pusch_config.dmrs.length,
                                        nDMRSAddPos=pusch_config.dmrs.additional_position,
                                        nPTRSPresent=pusch_config.My_Config.Ue[0].Ptrs,
                                        nAck=pusch_config.My_Config.Ue[0].OAck,
                                        nAlphaScaling=pusch_config.My_Config.Ue[0].ScalingFactor,
                                        nBetaOffsetACKIndex=pusch_config.My_Config.Ue[0].IHarqAckOffset,
                                        nCsiPart1=pusch_config.My_Config.Ue[0].OCsi1,
                                        nBetaOffsetCsiPart1Index=pusch_config.My_Config.Ue[0].ICsi1Offset,
                                        nCsiPart2=pusch_config.My_Config.Ue[0].OCsi2,
                                        nBetaOffsetCsiPart2Index=pusch_config.My_Config.Ue[0].ICsi2Offset,
                                        nTpPi2BPSK=pusch_config.My_Config.Ue[0].TpPi2Bpsk,
                                        nTPPuschID=pusch_config.My_Config.Ue[0].NRsId,
                                        nRxRUIdx=np.arange(0, pusch_config.My_Config.Sys.NRxAnt),
                                        nUE=1,
                                        nPduIdx=[0],
                                        Channel_model=f"{channel}-{model}",
                                        Speed=speed,
                                        Delay_spread=delay_spread,
                                        Esno_db=esno_db,
                                        Data_filename=timestamp,
                                        Data_dirname=name
                                    )
                                )
                    pbar.update(1)  # Increment progress

            df = pd.DataFrame.from_records(pusch_records, columns=PuschRecord._fields)
            if save_dataset: df.to_parquet(f'{parquet_dir}/{name}.parquet', engine="pyarrow")
    return df

In [38]:
assert False

AssertionError: 

# Case 4RB gen...

In [13]:
cfg_4 = MyConfig(SystemConfig(), [UeConfig()])
pusch_4 = MyPUSCHConfig(cfg_4)
pusch_4.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 246
n_size_grid : 273
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=246, FrequencyRange=1, BandWidth=100, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=273, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(TransformPrecoding=0, Rnti=20002, nId=246, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=31, NPrb=4, FrequencyHoppingMode=0, McsTable=0, Mcs=3, ILbrm=0, nScId=0, NnScIdId=246, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=1, PuschMappingType=0, DmrsTypeAPositi

In [14]:
pusch_4_2 = pusch_4.clone()
pusch_4_2.n_rnti = 20069
pusch_4_2.tb.n_id = 10
pusch_4_2.phy_cell_id = 123
pusch_4_2.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 123
n_size_grid : 273
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=246, FrequencyRange=1, BandWidth=100, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=273, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(TransformPrecoding=0, Rnti=20002, nId=246, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=31, NPrb=4, FrequencyHoppingMode=0, McsTable=0, Mcs=3, ILbrm=0, nScId=0, NnScIdId=246, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=1, PuschMappingType=0, DmrsTypeAPositi

In [15]:
# data_dir = '/content/dataset' # if colab
data_dir = '../Pusch_data/dataset' # if workspace

pusch_4_2 = pusch_4.clone()
pusch_4_2.n_rnti = 20069
pusch_4_2.My_Config.Ue[0].FirstPrb = 213
pusch_4_3 = pusch_4.clone()
pusch_4_3.n_rnti = 11235
pusch_4_3.My_Config.Ue[0].FirstPrb = 100
name = datetime.now().strftime("%Y%m%d%H%M%S%f")

start = time.time()
df = generate_data(name=f'{name}',
              data_dir=data_dir,
              pusch_configs=[
                  pusch_4,
                  # pusch_4_2,
                  # pusch_4_3
                  ],
              channel_scenarios=[
                  'CDL-A-150-1',
                  'CDL-B-150-1',
                  'CDL-C-150-1',
                  'CDL-D-150-1',
                  'CDL-E-150-1',
                  'CDL-A-10-10',
                  'CDL-B-10-10',
                  'CDL-C-10-10',
                  'CDL-D-10-10',
                  'CDL-E-10-10',
                  'CDL-A-50-4',
                  'CDL-B-50-4',
                  'CDL-C-50-4',
                  'CDL-D-50-4',
                  'CDL-E-50-4'
                  ],
              esno_dbs=[i for i in np.arange(5.,5.1,0.5)],
              slots=[4,5,14,15]*3,
              save_dataset=None
)
duration = time.time() - start

(config 0 | channel CDL-A-150-1 | 5.0 dB | slot 5 | Sample: 6/12:   3%|▎         | 5/180 [00:05<03:15,  1.12s/it] 


KeyboardInterrupt: 

In [ ]:
# !cp /content/dataset/parquet/{name}.parquet /content/drive/MyDrive/Pusch_data/dataset/parquet
# !cp /content/dataset/hdf5/{name}.hdf5 /content/drive/MyDrive/Pusch_data/dataset/hdf5

# Case 273 RB gen...

In [22]:
ue_273 = UeConfig(
                TransformPrecoding = 0,
                Rnti = 40007,
                nId = 443,
                CodeBookBased = 0,
                DmrsPortSetIdx = [0],
                NLayers = 1,
                NumDmrsCdmGroupsWithoutData = 2,
                Tpmi = 0,
                FirstSymb = 0,
                NPuschSymbAll = 14,
                RaType = 1,
                FirstPrb = 0,
                NPrb = 273,
                FrequencyHoppingMode = 0,
                McsTable = 0,
                Mcs = 3,
                ILbrm = 0,
                nScId = 0,
                NnScIdId = 443,
                DmrsConfigurationType = 0,
                DmrsDuration = 1,
                DmrsAdditionalPosition = 1,
                PuschMappingType = 0,
                DmrsTypeAPosition = 3,
                HoppingMode = 0,
                NRsId = 0,
                Ptrs = 0,
                ScalingFactor = 0,
                OAck = 0,
                IHarqAckOffset = 11,
                OCsi1 = 0,
                ICsi1Offset = 7,
                OCsi2 = 0,
                ICsi2Offset = 0,
                NPrbOh = 0,
                nCw = 1,
                TpPi2Bpsk = 0
            )
cfg_273 = MyConfig(
                SystemConfig(
                    NCellId = 443,
                    FrequencyRange = 1,
                    BandWidth = 100,
                    Numerology = 1,
                    CpType = 0,
                    NTxAnt = 1,
                    NRxAnt = 8,
                    BwpNRb = 273,
                    BwpRbOffset = 0,
                    harqProcFlag = 0,
                    nHarqProc = 1,
                    rvSeq = 0
                ),
                [ue_273] 
            )
pusch_273 = MyPUSCHConfig(cfg_273)
pusch_273.show()

Carrier Configuration
cyclic_prefix : normal
cyclic_prefix_length : 2.3437500000000002e-06
frame_duration : 0.01
frame_number : 0
kappa : 64.0
mu : 1
n_cell_id : 443
n_size_grid : 273
n_start_grid : 0
num_slots_per_frame : 20
num_slots_per_subframe : 2
num_symbols_per_slot : 14
slot_number : 4
sub_frame_duration : 0.001
subcarrier_spacing : 30
t_c : 5.086263020833334e-10
t_s : 3.2552083333333335e-08

PUSCH Configuration
My_Config : MyConfig(Sys=SystemConfig(NCellId=443, FrequencyRange=1, BandWidth=100, Numerology=1, CpType=0, NTxAnt=1, NRxAnt=8, BwpNRb=273, BwpRbOffset=0, harqProcFlag=0, nHarqProc=1, rvSeq=0), Ue=[UeConfig(TransformPrecoding=0, Rnti=40007, nId=443, CodeBookBased=0, DmrsPortSetIdx=[0], NLayers=1, NumDmrsCdmGroupsWithoutData=2, Tpmi=0, FirstSymb=0, NPuschSymbAll=14, RaType=1, FirstPrb=0, NPrb=273, FrequencyHoppingMode=0, McsTable=0, Mcs=3, ILbrm=0, nScId=0, NnScIdId=443, DmrsConfigurationType=0, DmrsDuration=1, DmrsAdditionalPosition=1, PuschMappingType=0, DmrsTypeAPosit

In [23]:
# Set a random seed for reproducibility
random.seed(42)
# Generate 200 unique PCI values (0-1023)
pci_values = random.sample(range(1024), 40)

# Generate 200 random RNTI values (0-65535)
rnti_values = [random.randint(0, 65535) for _ in range(40)]

In [24]:
pusch_config_273RB = []
for pci, rnti in zip(pci_values, rnti_values):
    pusch_instance = pusch_273.clone()
    pusch_instance.phy_cell_id = pci
    pusch_instance.n_rnti = rnti
    pusch_config_273RB.append(pusch_instance)

In [25]:
# data_dir = '/content/dataset' # if colab
data_dir = '../Pusch_data/dataset' # if workspace
name = datetime.now().strftime("%Y%m%d%H%M%S%f")
start = time.time()
df = generate_data(name=f'{name}',
              data_dir=data_dir,
              pusch_configs=[pusch_273],
              channel_scenarios=[
                  'CDL-A-150-1',
                  'CDL-B-150-1',
                  'CDL-C-150-1',
                  'CDL-D-150-1',
                  'CDL-E-150-1',
                  'CDL-A-10-10',
                  'CDL-B-10-10',
                  'CDL-C-10-10',
                  'CDL-D-10-10',
                  'CDL-E-10-10',
                  'CDL-A-50-4',
                  'CDL-B-50-4',
                  'CDL-C-50-4',
                  'CDL-D-50-4',
                  'CDL-E-50-4'
                  ],
              esno_dbs=[i for i in np.arange(5.,-5.1,-0.5)],
              slots=[
                  4,
                  5,
                  14,
                  15
                  ]*41,
              save_dataset=None
)
duration = time.time() - start

(config 0 | channel CDL-E-150-1 | 5.0 dB | slot 5 | Sample: 2/164:   0%|          | 1/3444 [00:09<9:03:32,  9.47s/it]

tf.Tensor(
[[[ 9.0296823e-01+8.72570455e-01j -4.8863450e-01-6.89292729e-01j
    7.8497422e-01-9.72536385e-01j ... -1.4999557e-01+1.17953289e+00j
   -2.5447989e-01+9.98702049e-01j  1.3749704e+00+2.07332224e-01j]
  [-1.0205951e+00+7.54897654e-01j  5.1861048e-01-9.49026525e-01j
   -8.3181012e-01-5.19862652e-01j ... -1.0331440e+00-3.49052548e-02j
    4.3161511e-01-2.93306291e-01j  1.0264136e+00+4.09982145e-01j]
  [ 1.0501184e+00-3.85237515e-01j -8.1126738e-01+7.32223272e-01j
    6.4059949e-01+1.17278576e+00j ... -9.9097115e-01-1.18148506e-01j
    8.7369150e-01-7.55072951e-01j  1.3126286e+00+1.37472361e-01j]
  ...
  [-1.0059856e+00+1.26378012e+00j -3.5430333e-01-1.84470266e-01j
    1.1177832e+00-1.34646416e+00j ...  1.0033268e+00-4.70802456e-01j
    1.3889062e+00-1.40495837e+00j -2.1580805e-01-1.31870613e-01j]
  [ 8.2570338e-01+6.47168517e-01j  6.2989151e-01+1.63372397e+00j
   -5.5115426e-01+3.61644804e-01j ... -1.0574713e+00+1.74862731e+00j
    9.7353673e-01-1.43676519e+00j  3.4607849e-01+

(config 0 | channel CDL-E-150-1 | 5.0 dB | slot 14 | Sample: 3/164:   0%|          | 2/3444 [00:10<4:07:55,  4.32s/it]

tf.Tensor(
[[[ 1.0613214 -1.3829567j  -1.1013778 -0.92952335j
   -1.0112197 -1.2541771j  ... -0.25512928+0.42410764j
    1.1686139 +0.6099378j  -0.50144905-1.2538223j ]
  [ 0.8784045 -0.16933542j  0.35420495+0.13422585j
   -1.0893286 -0.69903606j ...  0.03467178+0.713329j
   -0.680764  -0.8123609j   0.7620687 -0.553295j  ]
  [ 1.4287024 +1.1251765j   0.49754474-0.01594985j
   -1.0869275 -0.9747029j  ...  0.39575306+0.0185042j
   -0.5860789 +0.8710575j  -1.3825216 -0.85332763j]
  ...
  [-1.1805382 -0.906949j   -0.1672032 -0.39590296j
    0.8587944 +0.99449277j ... -1.0293492 +0.06408655j
    1.0502787 +1.5088769j  -0.5670751 -0.11827411j]
  [ 0.94565135+0.86821926j  0.30146962+1.4389318j
   -0.8425403 -0.36711517j ...  0.20607114+0.83637786j
    0.9078161 +0.38928175j -1.364419  -0.48894933j]
  [-0.5041704 -0.54363656j  0.6099759 -0.7621819j
   -1.0724579 +0.31109148j ...  0.7455448 -1.0694313j
    0.695223  +1.0099438j  -0.82966053-0.0081386j ]]

 [[ 1.4456214 -0.87985766j -1.0087464 -

(config 0 | channel CDL-E-150-1 | 5.0 dB | slot 15 | Sample: 4/164:   0%|          | 3/3444 [00:10<2:32:38,  2.66s/it]

...

tf.Tensor(
[[[-0.38164774+0.3388898j   0.61453074-0.28083402j
    0.6270234 -0.9084382j  ...  0.3830737 -1.1593559j
    0.8656792 +0.19949365j -0.9629071 -0.4060988j ]
  [-1.2825043 +0.71303874j  0.630626  +0.27258444j
    1.7073824 -0.3775771j  ...  0.16480607+1.4617159j
   -0.66361773+0.23556685j  0.5807021 +0.9858967j ]
  [ 0.59594095-0.389167j   -0.6156286 +0.695138j
   -0.9851854 +1.11795j    ... -0.47179154+0.1316275j
    1.4778349 -0.4822361j   0.33626768+0.5270399j ]
  ...
  [-1.3269744 -1.1105154j  -0.6697666 -0.21333914j
    1.1647863 +1.1986682j  ... -0.19540766-0.56457907j
   -1.2321074 -0.05798244j -0.07953541+0.32881486j]
  [ 1.0991373 -1.1678939j   0.03308612+0.53572726j
   -1.0199566 -1.342843j   ... -1.3996434 +1.0374708j
    0.4633466 +1.2596513j  -0.7356507 -0.501845j  ]
  [ 0.30616793+0.08900321j -0.5333794 -0.84587896j
   -0.7525913 +0.46911824j ... -0.38929296+0.6507729j
    1.1582718 +0.05958885j -0.7455156 -0.0165478j ]]

 [[-1.1831644 +0.6951265j   1.3376834 -0

(config 0 | channel CDL-E-150-1 | 5.0 dB | slot 4 | Sample: 21/164:   1%|          | 20/3444 [00:23<40:42,  1.40it/s] 

tf.Tensor(
[[[-0.8811077 +8.01854730e-01j  0.6184618 +1.41297460e+00j
    0.94836307-3.19510877e-01j ... -0.6230508 +1.16830468e-01j
    0.5565606 +8.01666915e-01j -1.0715017 -1.31001627e+00j]
  [ 0.6304081 +8.81270230e-01j  0.86646765-5.60135067e-01j
    0.6029705 +1.07178760e+00j ...  0.49950328-1.16407871e+00j
    0.8299088 -9.09575760e-01j -0.60006243+1.10271621e+00j]
  [ 0.5594255 -7.81605005e-01j -0.90345895+1.06059134e-01j
   -0.01149559+5.78061461e-01j ...  0.96648365+9.00298655e-02j
   -0.51245314+1.15802896e+00j  0.5012241 -2.39552081e-01j]
  ...
  [-1.3632681 +6.02664232e-01j  0.36793804+3.19439769e-02j
    0.8313089 +1.17450988e+00j ... -0.5562719 -1.77487493e-01j
    0.57899904-8.05760026e-01j  0.17823756+6.18932024e-02j]
  [ 1.3408575 -6.26903355e-01j -0.49626982+7.78987050e-01j
   -0.89098763+9.66965437e-01j ... -0.35217038+3.86676639e-01j
    0.8045981 -1.23530066e+00j  0.8764122 +4.67684269e-01j]
  [ 0.42849454+4.87053484e-01j  1.2842057 +7.92115927e-03j
    0.7984069 

(config 0 | channel CDL-E-150-1 | 5.0 dB | slot 4 | Sample: 21/164:   1%|          | 20/3444 [00:23<1:08:12,  1.20s/it]


KeyboardInterrupt: 

In [17]:
# data_dir = '/content/dataset' # if colab
data_dir = '../Pusch_data/dataset' # if workspace
name = datetime.now().strftime("%Y%m%d%H%M%S%f")
start = time.time()
df = generate_data(name=f'{name}',
              data_dir=data_dir,
              pusch_configs=[pusch_273, *pusch_config_273RB],
              channel_scenarios=[
                  'CDL-A-150-1',
                  'CDL-B-150-1',
                  'CDL-C-150-1',
                  'CDL-D-150-1',
                  'CDL-E-150-1',
                  'CDL-A-10-10',
                  'CDL-B-10-10',
                  'CDL-C-10-10',
                  'CDL-D-10-10',
                  'CDL-E-10-10',
                  'CDL-A-50-4',
                  'CDL-B-50-4',
                  'CDL-C-50-4',
                  'CDL-D-50-4',
                  'CDL-E-50-4'
                  ],
              esno_dbs=[i for i in np.arange(5.,-5.1,-0.5)],
              slots=[
                  4,
                  5,
                  14,
                  15
                  ],
              save_dataset=None
)
duration = time.time() - start

Generating Data:   0%|          | 0/3444 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [61]:
# !cp /content/dataset/parquet/{name}.parquet /content/drive/MyDrive/Pusch_data/dataset/parquet
# !cp /content/dataset/hdf5/{name}.hdf5 /content/drive/MyDrive/Pusch_data/dataset/hdf5

In [16]:
assert False

AssertionError: 

In [ ]:
# !du -sh /content/dataset/pickle/20250304165249747666

129M	/content/dataset/pickle/20250304165249747666


In [ ]:
# !du -sh /content/dataset/hdf5/20250304171421232817.hdf5

113M	/content/dataset/hdf5/20250304171421232817.hdf5


In [27]:
def load_hdf5(parent_name, group_name):
    with h5py.File(f'{parent_name}.hdf5', "r") as f:
        b = f[f"{group_name}_b"][:]
        c = f[f"{group_name}_c"][:]
        y = f[f"{group_name}_y"][:]
    return b, c, y

def load_pickle(parent_name, group_name):
    """Saves data to a pickle file."""
    def load_from_pickle(filename):
        with open(filename, "rb") as f:
            return pickle.load(f)

    b = load_from_pickle(f'{parent_name}/{group_name}.b.pkl')
    c = load_from_pickle(f'{parent_name}/{group_name}.c.pkl')
    y = load_from_pickle(f'{parent_name}/{group_name}.y.pkl')

    return b, c, y

In [ ]:
def abc():
    df = pd.read_parquet('/content/dataset/parquet/20250304171421232817.parquet', engine='pyarrow')
    for n, row in enumerate(df.itertuples(index=False)):
        data_filename = row.Data_filename
        data_dirname = row.Data_dirname
        b, c, y = load_hdf5(f'/content/dataset/hdf5/{data_dirname}', data_filename)
        if n % 1000 == 0: print(b.shape, b.dtype, c.shape, c.dtype, y.shape, y.dtype)
# %timeit abc()

(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,)

In [ ]:
def bcd():
    df = pd.read_parquet('/content/dataset/parquet/20250304165249747666.parquet', engine='pyarrow')
    for n, row in enumerate(df.itertuples(index=False)):
        data_filename = row.Data_filename
        data_dirname = row.Data_dirname
        b, c, y = load_pickle(f'/content/dataset/hdf5/{data_dirname}', data_filename)
        if n % 1000 == 0: print(b.shape, b.dtype, c.shape, c.dtype, y.shape, y.dtype)
# %timeit abc()

(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,) uint8 (8, 14, 48) complex64
(288,) uint8 (1152,)

In [ ]:
# %timeit !cp /content/dataset/hdf5/20250304171421232817.hdf5 /content/drive/MyDrive/Pusch_data/dataset/hdf5


1.34 s ± 379 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# start = time.time()
# !cp -r /content/dataset/pickle/20250304165249747666 /content/drive/MyDrive/Pusch_data/dataset/pickle
# duration = time.time() - start
# duration

94.83374857902527

In [ ]:
# start = time.time()
# !cp -r /content/drive/MyDrive/Pusch_data/dataset/pickle/20250304165249747666 /content/dataset_copy/pickle
# duration = time.time() - start
# duration

272.0682830810547

In [ ]:
# %timeit !cp /content/drive/MyDrive/Pusch_data/dataset/hdf5/20250304171421232817.hdf5 /content/dataset_copy/hdf5

The slowest run took 9.36 times longer than the fastest. This could mean that an intermediate result is being cached.
1.2 s ± 1.44 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# assert False

In [ ]:
# !cp /content/drive/MyDrive/Pusch_data/dataset/parquet/20250304181811698207.parquet /content/dataset/parquet

In [ ]:
# !cp /content/drive/MyDrive/Pusch_data/dataset/hdf5/20250304181811698207.hdf5 /content/dataset/hdf5

In [ ]:
# !gdown --folder --id 1-D1FSox1TpjKW59wPqpZvbH1b3Tcghk6 -O /content/abc

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Retrieving folder contents
Processing file 1-Qpl2BqeVPY75whKPPx8ePn__8PoCWc7 20250304171421232817.hdf5
Processing file 1FxvsTX-bYFQHFMk6r4Vrx0tl7vBnOVC2 20250304181811698207.hdf5
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1-Qpl2BqeVPY75whKPPx8ePn__8PoCWc7
From (redirected): https://drive.google.com/uc?id=1-Qpl2BqeVPY75whKPPx8ePn__8PoCWc7&confirm=t&uuid=5939bbc5-ad30-429c-899a-960d9f7081fb
To: /content/abc/20250304171421232817.hdf5
100% 118M/118M [00:01<00:00, 74.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1FxvsTX-bYFQHFMk6r4Vrx0tl7vBnOVC2
From (redirected): https://drive.google.com/uc?id=1FxvsTX-bYFQHFMk6r4Vrx0tl7vBnOVC2

In [ ]:
# df = pd.read_parquet('/content/dataset/parquet/20250304181811698207.parquet', engine='pyarrow')

In [ ]:
# df = pd.read_parquet('/content/dataset/parquet/20250304181811698207.parquet', engine='pyarrow')
# for n, row in enumerate(df.itertuples(index=False)):
#     data_filename = row.Data_filename
#     data_dirname = row.Data_dirname
#     b, c, y = load_hdf5(f'/content/dataset/hdf5/{data_dirname}', data_filename)
#     if n % 1000 == 0: print(b.shape, b.dtype, c.shape, c.dtype, y.shape, y.dtype)

(19464,) uint8 (78624,) uint8 (8, 14, 3276) complex64
(19464,) uint8 (78624,) uint8 (8, 14, 3276) complex64
(19464,) uint8 (78624,) uint8 (8, 14, 3276) complex64


NameError: name 'start' is not defined